# 04-1. 원본 데이터 노이즈 후보 분류

원본을 수정하지 않고 전처리 후보를 유형별 CSV로 저장한다. repeat_4gram은 같은 정규화 4어절이 3회 이상 등장하는 경우다.

In [1]:
import os, re, sys
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
RESULTS = PROJECT / "review_data" / "04-1_raw_candidates"
RESULTS.mkdir(parents=True, exist_ok=True)
word_pattern = re.compile(r"[가-힣A-Za-z0-9]+(?:[.,]\d+)?")
print("결과 폴더:", RESULTS)

결과 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-1_raw_candidates


In [2]:
revision = HfApi().dataset_info(DATASET_ID).sha
dataset = load_dataset(DATASET_ID, revision=revision)
frames=[]
for split in ["train", "validation", "test"]:
    frame=dataset[split].to_pandas().copy()
    frame.insert(0, "source_index", np.arange(len(frame)))
    frame.insert(0, "split", split)
    frame.insert(0, "row_id", split + ":" + frame.source_index.astype(str))
    frames.append(frame)
df=pd.concat(frames, ignore_index=True)

def words(text): return word_pattern.findall(str(text).lower())
def max_4gram(text):
    tokens=words(text); counts=Counter(tuple(tokens[i:i+4]) for i in range(len(tokens)-3))
    return max(counts.values(), default=0)
def overlap(document, summary):
    source=set(words(document)); target=words(summary)
    return np.nan if not target else sum(token in source for token in target)/len(target)
def numbers(text):
    text=re.sub(r"(?<=\d)\s+(?=\d)", "", str(text)).replace(",", "")
    return set(re.findall(r"\d+(?:\.\d+)?", text))

df["document_words"]=df.document.map(lambda x: len(words(x)))
df["document_chars"]=df.document.str.len(); df["summary_chars"]=df.summary.str.len()
df["char_compression_ratio"]=df.summary_chars/df.document_chars
df["max_same_4gram_occurrences"]=df.summary.map(max_4gram)
df["repeat_4gram"]=df.max_same_4gram_occurrences.ge(3)
df["summary_document_overlap"]=[overlap(d,s) for d,s in zip(df.document,df.summary)]
df["low_overlap"]=df.summary_document_overlap.lt(.30)
df["extreme_ratio"]=(df.char_compression_ratio.lt(.02)|df.char_compression_ratio.gt(.80))
df["summary_longer"]=df.summary_chars.gt(df.document_chars)
df["short_document"]=df.document_words.lt(30)
doc_numbers=df.document.map(numbers); sum_numbers=df.summary.map(numbers)
df["numbers_only_in_summary"]=[", ".join(sorted(s-d)) for d,s in zip(doc_numbers,sum_numbers)]
df["number_mismatch"]=df.numbers_only_in_summary.ne("")
df["has_url"]=df.document.str.contains(r"https?://|www\.",case=False,regex=True)
df["has_boilerplate"]=df.document.str.contains(r"무단\s*(?:전재|복제|배포)|재배포\s*금지|인터뷰\s*자료의\s*저작권|실제\s*방송\s*내용과\s*차이",regex=True)
df["replacement_char"]=df.document.str.contains("�",regex=False)
df["exact_document_summary_duplicate"]=df.duplicated(["document","summary"],keep=False)
summary_doc_n=df.groupby("summary").document.transform("nunique")
df["shared_summary_different_document"]=summary_doc_n.gt(1)


In [3]:
BASE=["row_id","split","source_index","date","category","press","title","link","document","summary","document_words","char_compression_ratio","summary_document_overlap","max_same_4gram_occurrences","numbers_only_in_summary"]
RULES={
 "01_repeat_4gram":df.repeat_4gram, "02_low_overlap":df.low_overlap,
 "03_extreme_ratio":df.extreme_ratio, "04_summary_longer":df.summary_longer,
 "05_number_mismatch":df.number_mismatch, "06_short_document":df.short_document,
 "07_url":df.has_url, "08_boilerplate":df.has_boilerplate,
 "09_replacement_char":df.replacement_char,
 "10_exact_document_summary_duplicate":df.exact_document_summary_duplicate,
 "11_shared_summary_different_document":df.shared_summary_different_document}
rows=[]
for name,mask in RULES.items():
    out=df.loc[mask,BASE].copy(); out["review_decision"]=""; out["memo"]=""
    out.to_csv(RESULTS/f"{name}.csv",index=False,encoding="utf-8-sig")
    rows.append({"file":f"{name}.csv","rows":len(out)})
summary=pd.DataFrame(rows); summary.to_csv(RESULTS/"00_export_summary.csv",index=False,encoding="utf-8-sig")
pd.DataFrame([{"dataset_id":DATASET_ID,"dataset_revision":revision,"policy":"read-only candidate export"}]).to_csv(RESULTS/"00_run_info.csv",index=False,encoding="utf-8-sig")
display(summary)

,file,rows
0,01_repeat_4gram.csv,1376
1,02_low_overlap.csv,6
2,03_extreme_ratio.csv,2823
3,04_summary_longer.csv,2132
4,05_number_mismatch.csv,777
5,06_short_document.csv,1514
6,07_url.csv,83
7,08_boilerplate.csv,53
8,09_replacement_char.csv,4
9,10_exact_document_summary_duplicate.csv,3785
